In [1]:
import yfinance as yf
print(f"yfinance version: {yf.__version__}")
print(f"Loaded from: {yf.__file__}")

yfinance version: 1.3.0
Loaded from: /Users/nitishkumar/stockwatch/.venv/lib/python3.13/site-packages/yfinance/__init__.py


In [2]:
# Fetch Reliance Industries (NSE listing)
stock = yf.Ticker("RELIANCE.NS")

# Get the most recent price info
info = stock.info
print(f"Name: {info.get('longName')}")
print(f"Current price: ₹{info.get('currentPrice')}")
print(f"52-week high: ₹{info.get('fiftyTwoWeekHigh')}")
print(f"52-week low: ₹{info.get('fiftyTwoWeekLow')}")
print(f"Day range: ₹{info.get('regularMarketDayLow')} - ₹{info.get('regularMarketDayHigh')}")

Name: Reliance Industries Limited
Current price: ₹1336.4
52-week high: ₹1611.8
52-week low: ₹1290.0
Day range: ₹1329.2 - ₹1364.8


In [3]:
# Get 60 days of daily history
history = stock.history(period="60d")
print(f"Number of trading days: {len(history)}")
print()
print("Last 5 days:")
print(history.tail()[['Open', 'Close', 'High', 'Low', 'Volume']])

Number of trading days: 60

Last 5 days:
                                  Open        Close         High          Low  \
Date                                                                            
2026-05-11 00:00:00+05:30  1420.000000  1388.199951  1428.000000  1382.000000   
2026-05-12 00:00:00+05:30  1392.000000  1364.000000  1393.500000  1360.300049   
2026-05-13 00:00:00+05:30  1361.400024  1358.800049  1372.400024  1352.400024   
2026-05-14 00:00:00+05:30  1365.199951  1361.800049  1378.000000  1358.400024   
2026-05-15 00:00:00+05:30  1356.800049  1336.400024  1364.800049  1329.199951   

                             Volume  
Date                                 
2026-05-11 00:00:00+05:30  15261787  
2026-05-12 00:00:00+05:30  24357500  
2026-05-13 00:00:00+05:30  13797989  
2026-05-14 00:00:00+05:30  17303059  
2026-05-15 00:00:00+05:30  19976192  


In [4]:
import pandas as pd

# Your watchlist
watchlist = {
    "RELIANCE.NS": "Reliance Industries",
    "INDIGO.NS": "InterGlobe Aviation",
    "HINDUNILVR.NS": "Hindustan Unilever",
    "ASIANPAINT.NS": "Asian Paints",
    "HDFCBANK.NS": "HDFC Bank",
}

# Fetch a year of data for all of them in one call
tickers = list(watchlist.keys())
data = yf.download(tickers, period="1y", auto_adjust=True, progress=False)

print(f"Fetched data shape: {data.shape}")
print()
print("Columns (first 6):")
print(data.columns[:6])
print()
print("Last 3 days of close prices:")
print(data['Close'].tail(3))

Fetched data shape: (250, 25)

Columns (first 6):
MultiIndex([('Close', 'ASIANPAINT.NS'),
            ('Close',   'HDFCBANK.NS'),
            ('Close', 'HINDUNILVR.NS'),
            ('Close',     'INDIGO.NS'),
            ('Close',   'RELIANCE.NS'),
            ( 'High', 'ASIANPAINT.NS')],
           names=['Price', 'Ticker'])

Last 3 days of close prices:
Ticker      ASIANPAINT.NS  HDFCBANK.NS  HINDUNILVR.NS    INDIGO.NS  \
Date                                                                 
2026-05-13    2617.600098   749.599976    2267.300049  4255.799805   
2026-05-14    2622.199951   769.549988    2248.699951  4280.500000   
2026-05-15    2605.600098   767.500000    2272.199951  4314.899902   

Ticker      RELIANCE.NS  
Date                     
2026-05-13  1358.800049  
2026-05-14  1361.800049  
2026-05-15  1336.400024  


In [5]:
# Get the Close prices in a clean DataFrame (rows = dates, columns = tickers)
close = data['Close']

# Compute 52-week high/low for each stock
fifty_two_week_high = close.max()
fifty_two_week_low = close.min()
current_price = close.iloc[-1]  # most recent close

# Compute position within the 52w range (0 = at 52w low, 1 = at 52w high)
position = (current_price - fifty_two_week_low) / (fifty_two_week_high - fifty_two_week_low)

# Compute 7-day and 30-day percent change
change_7d = (current_price - close.iloc[-8]) / close.iloc[-8] * 100
change_30d = (current_price - close.iloc[-31]) / close.iloc[-31] * 100

# Assemble a summary table
summary = pd.DataFrame({
    "Name": [watchlist[t] for t in close.columns],
    "Current": current_price.round(2),
    "52w Low": fifty_two_week_low.round(2),
    "52w High": fifty_two_week_high.round(2),
    "52w Position": (position * 100).round(1).astype(str) + "%",
    "7d Change": change_7d.round(2).astype(str) + "%",
    "30d Change": change_30d.round(2).astype(str) + "%",
})

print(summary)

                              Name  Current  52w Low  52w High 52w Position  \
Ticker                                                                        
ASIANPAINT.NS         Asian Paints   2605.6  2121.30    2968.5        57.2%   
HDFCBANK.NS              HDFC Bank    767.5   731.55    1012.9        12.8%   
HINDUNILVR.NS   Hindustan Unilever   2272.2  2052.20    2671.6        35.5%   
INDIGO.NS      InterGlobe Aviation   4314.9  3943.50    6155.5        16.8%   
RELIANCE.NS    Reliance Industries   1336.4  1304.60    1592.3        11.1%   

              7d Change 30d Change  
Ticker                              
ASIANPAINT.NS     3.44%     17.06%  
HDFCBANK.NS      -3.65%       3.4%  
HINDUNILVR.NS    -1.94%     10.05%  
INDIGO.NS        -4.54%      3.21%  
RELIANCE.NS      -7.06%      -2.4%  
